# DINOv3 Patch Embedding Extraction

This notebook extracts spatial patch embeddings from cropped culvert CCTV images using a pretrained DINOv3 ViT-L/16 visual encoder.

## Purpose

The extracted patch representations preserve local spatial information from the monitored trash-screen region and can subsequently be used for downstream blockage analysis. Each image is cropped using site-specific coordinates, resized to 224 × 224 pixels, passed through the frozen DINOv3 visual encoder, and represented by its patch-token embeddings.

For a 224 × 224 input and a 16 × 16 patch size, the model produces a 14 × 14 spatial grid containing 196 patch tokens. Each patch is represented by a 1,024-dimensional feature vector.

## Outputs

For each monitoring site, the notebook saves:

- `dinov3_patches_<site>.npy` — patch embeddings with shape `(n_images, 196, 1024)`.
- `meta_<site>.csv` — image filename, class label, site and source filepath aligned with the first dimension of the embedding array.
- `errors_<site>.csv` — extraction errors, when present.

Embeddings are stored as `float16` to reduce disk usage. The final section reloads and validates the saved arrays before downstream modelling.

> **Environment:** The notebook was developed in Google Colab and expects the dataset, crop-coordinate file, DINOv3 repository and pretrained weights to be configured locally using the paths defined below.


## 1. Environment and dependencies

Import the libraries required for data handling, image preprocessing and DINOv3 inference, then mount Google Drive.


In [ ]:
import sys
import os
import gc
import pandas as pd
import numpy as np
import torch

from pathlib import Path
from PIL import Image
from torchvision import transforms

# Optional: mount Google Drive when running in Google Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

## 2. Paths and output directory

Define the dataset, sampled-image index, site-specific crop coordinates, model resources and destination for extracted embeddings.


In [ ]:
from pathlib import Path

# ------------------------------------------------------------------
# Repository configuration
# ------------------------------------------------------------------
# Set PROJECT_ROOT
PROJECT_ROOT = Path(".")

# Expected local resources. .
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "dinov3_patch_embeddings"

SAMPLE_CSV = DATA_DIR / "sample_4000.csv"
CROP_COORDS = DATA_DIR / "crop_coordinates.csv"
DATASET_ROOT = DATA_DIR / "images"

# Local DINOv3 resources.
DINOV3_REPO = MODEL_DIR / "dinov3"
DINOV3_WEIGHTS = MODEL_DIR / "dinov3_weights.pth"
DINOTXT_WEIGHTS = MODEL_DIR / "dinotxt_weights.pth"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Load the sampled dataset

Load the balanced 4,000-image study sample and inspect its basic dimensions.


In [ ]:
# Cell 3: Load dataset

sampled_df = pd.read_csv(CSV_PATH)

print(f"Total images: {len(sampled_df)}")
print(f"Total sites: {sampled_df['site'].nunique()}")

display(sampled_df.head())

## 4. Load site-specific crop coordinates

Read the crop definitions used to isolate the monitored trash-screen region from each CCTV image.


In [ ]:
# Cell 3a: Load crop coordinates

crop_df = pd.read_csv(
    CROP_TXT_PATH,
    sep=r'\s+',
    skiprows=4,
    names=[
        'location',
        'x_min',
        'x_max',
        'y_min',
        'y_max'
    ]
)

crop_df = crop_df.set_index('location')

print(f"Crop coordinates loaded: {len(crop_df)} sites")

### 4.1 Resolve site-name mismatches

A small number of site identifiers differ between the sampled dataset and the crop-coordinate file. The mapping below reconciles those known naming differences.


In [ ]:
#Cell4: Name mismatch correction

crop_name_mapping = {
    'Devon_BarnstapleConeyGut_Scree':
        'Devon_BarnstapleConeyGut_Screen',

    'Cornwall_Mevagissey_PreScree':
        'Cornwall_Mevagissey_PreScreen',

    'Devon_LympstoneScree':
        'Devon_LympstoneScreen'
}

In [ ]:
def get_crop_coords(site):
    crop_name = crop_name_mapping.get(site, site)

    if crop_name in crop_df.index:
        row = crop_df.loc[crop_name]

        return (
            int(row['x_min']),
            int(row['x_max']),
            int(row['y_min']),
            int(row['y_max'])
        )

    return None

### 4.2 Validate crop coverage

Confirm that every monitoring site in the sampled dataset has a corresponding crop definition before feature extraction begins.


In [ ]:
# Check that all sampled sites have crop coordinates

sites = sampled_df['site'].unique()

missing_crops = []

for site in sites:
    coords = get_crop_coords(site)

    if coords is None:
        missing_crops.append(site)
        print(f"MISSING: {site}")
    else:
        print(f"OK: {site} -> {coords}")

print(f"\nSites without crop coordinates: {len(missing_crops)}")

## 5. Image preprocessing

Crop each image to the site-specific region of interest, resize it to 224 × 224 pixels and apply ImageNet normalisation before inference.


In [ ]:
#Cell 5: Preprocessing

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def load_image(filepath, site):
    img = Image.open(filepath).convert('RGB')

    coords = get_crop_coords(site)

    if coords is not None:
        x_min, x_max, y_min, y_max = coords

        img = img.crop(
            (
                x_min,
                y_min,
                x_max,
                y_max
            )
        )

    return preprocess(img)

### 5.1 Visual preprocessing check

Inspect one example to verify that the crop coordinates isolate the intended region and that the final model input matches the extraction pipeline.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

# Select one Corsham image
site_to_view = "sites_corshamaqueduct_cam1"

row = (
    sampled_df[
        sampled_df["site"] == site_to_view
    ]
    .sample(1, random_state=42)
    .iloc[0]
)

# Reconstruct the Colab path exactly as the extraction pipeline does
filepath = (
    Path(DATASET_FOLDER)
    / row["site"]
    / row["ground_truth"]
    / row["filename"]
)

# Load original
img = Image.open(filepath).convert("RGB")

# Apply the same crop coordinates used during extraction
coords = get_crop_coords(row["site"])

if coords is None:
    raise ValueError(
        f"No crop coordinates found for {row['site']}"
    )

x_min, x_max, y_min, y_max = coords

cropped = img.crop(
    (x_min, y_min, x_max, y_max)
)

# Resize exactly as the preprocessing pipeline does
resized = cropped.resize((224, 224))

# Plot original, crop and model input
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img)
axes[0].add_patch(
    patches.Rectangle(
        (x_min, y_min),
        x_max - x_min,
        y_max - y_min,
        linewidth=3,
        edgecolor="red",
        facecolor="none"
    )
)
axes[0].set_title("Original image with crop boundary")
axes[0].axis("off")

axes[1].imshow(cropped)
axes[1].set_title("Cropped region")
axes[1].axis("off")

axes[2].imshow(resized)
axes[2].set_title("224 × 224 input to DINOv3")
axes[2].axis("off")

plt.tight_layout()
plt.show()

print("File:", filepath)
print("Site:", row["site"])
print("Label:", row["ground_truth"])
print("Original size:", img.size)
print("Crop coordinates:", coords)
print("Crop size:", cropped.size)
print("Model input size:", resized.size)

## 6. Load the pretrained DINOv3 model

Install the lightweight text dependencies required by the local DINOv3 repository, then load the pretrained DINOv3 ViT-L/16 model and its DINO-TXT weights. Only the visual encoder is used for patch extraction.


In [ ]:
!pip install ftfy regex -q

In [ ]:
# Cell 7: Load DINOv3
# Model locations are defined in the configuration cell above.
DINOV3_REPO = str(DINOV3_REPO)
BACKBONE = str(DINOV3_WEIGHTS)
DINOTXT = str(DINOTXT_WEIGHTS)

if DINOV3_REPO not in sys.path:
    sys.path.append(DINOV3_REPO)

from dinov3.hub.dinotxt import (
    dinov3_vitl16_dinotxt_tet1280d20h24l
)


In [ ]:
model, tokenizer = (
    dinov3_vitl16_dinotxt_tet1280d20h24l(
        backbone_weights=BACKBONE,
        weights=DINOTXT
    )
)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

model = model.to(device)
model.eval()

print(f"Device: {device}")
print("DINO TXT loaded successfully")

## 7. Single-image extraction test

Run one image through the preprocessing pipeline and inspect the encoder outputs before processing the complete dataset.


In [ ]:
#Cell 8: Test on single image before full extraction

test_row = sampled_df.iloc[0]

test_filepath = (
    Path(DATASET_FOLDER)
    / test_row['site']
    / test_row['ground_truth']
    / test_row['filename']
)

test_image = load_image(
    test_filepath,
    test_row['site']
)

test_image = (
    test_image
    .unsqueeze(0)
    .to(device)
)

print(f"Input shape: {test_image.shape}")
print(f"Test image: {test_filepath}")

In [ ]:
with torch.inference_mode():
    outputs = (
        model.visual_model
        .get_class_and_patch_tokens(test_image)
    )

print(f"Number of outputs: {len(outputs)}")

for index, output in enumerate(outputs):
    if torch.is_tensor(output):
        print(
            f"Output {index}: "
            f"shape={tuple(output.shape)}, "
            f"dtype={output.dtype}"
        )
    else:
        print(
            f"Output {index}: "
            f"type={type(output)}"
        )

### 7.1 Separate global and local representations

The encoder returns a CLS token representing the image globally and patch tokens retaining spatially local visual information. This notebook stores the patch tokens.


In [ ]:
# Separate CLS and patch-token outputs
cls_token = outputs[0]
patch_tokens = outputs[1]

print(f"CLS token: {cls_token.shape}")
print(f"Patch tokens: {patch_tokens.shape}")

### 7.2 Validate patch-token structure

Check that the returned tensors have the expected batch, token and feature dimensions.


In [ ]:
# Cell 9: Validate patch-token dimensions
assert cls_token.ndim == 2, (
    f"Expected CLS tensor with 2 dimensions, "
    f"received {cls_token.shape}"
)

assert patch_tokens.ndim == 3, (
    f"Expected patch tensor with 3 dimensions, "
    f"received {patch_tokens.shape}"
)

assert patch_tokens.shape[0] == 1, (
    "The first patch dimension should be the batch size."
)

assert patch_tokens.shape[1] > 1, (
    "Only one token was returned. "
    "This does not appear to be a patch tensor."
)

print("Patch tensor validation passed")
print(f"Patches per image: {patch_tokens.shape[1]}")
print(f"Features per patch: {patch_tokens.shape[2]}")

### 7.3 Check storage representation

Convert one image's patch embeddings to `float16` and estimate the resulting storage requirement.


In [ ]:
#Cell 10: Saving patches for one image
test_patches = (
    patch_tokens
    .squeeze(0)
    .detach()
    .cpu()
    .to(torch.float16)
    .numpy()
)

print(f"Saved array shape: {test_patches.shape}")
print(f"Saved array type: {test_patches.dtype}")
print(f"Size in MB: {test_patches.nbytes / 1_000_000:.2f}")

## 8. Patch-extraction function

Encapsulate preprocessing and inference so that the same procedure is applied consistently to every image.


In [ ]:
# Cell 11: Patch-embedding extraction function
def extract_patch_embeddings(filepath, site):
    image = load_image(
        filepath,
        site
    )

    image = (
        image
        .unsqueeze(0)
        .to(device)
    )

    with torch.inference_mode():
        outputs = (
            model.visual_model
            .get_class_and_patch_tokens(image)
        )

        patch_tokens = outputs[1]

    if patch_tokens.ndim != 3:
        raise ValueError(
            f"Unexpected patch shape: "
            f"{patch_tokens.shape}"
        )

    patches = (
        patch_tokens
        .squeeze(0)
        .detach()
        .cpu()
        .to(torch.float16)
        .numpy()
    )

    return patches


In [ ]:
# Test the extraction function
test_patches = extract_patch_embeddings(
    test_filepath,
    test_row['site']
)

print(f"Function output: {test_patches.shape}")
print(f"Data type: {test_patches.dtype}")

## 9. End-to-end test on one site

Before running all ten sites, extract embeddings for one complete site, retain aligned metadata and record any image-level failures.


In [ ]:
# Cell 12: Extract one complete site as an end-to-end check
test_site = sampled_df['site'].unique()[0]

test_site_df = (
    sampled_df[
        sampled_df['site'] == test_site
    ]
    .reset_index(drop=True)
)

print(f"Test site: {test_site}")
print(f"Images: {len(test_site_df)}")

In [ ]:
patch_embeddings = []
metadata_records = []
error_records = []

for row_index, row in test_site_df.iterrows():

    filepath = (
        Path(DATASET_FOLDER)
        / row['site']
        / row['ground_truth']
        / row['filename']
    )

    try:
        patches = extract_patch_embeddings(
            filepath,
            row['site']
        )

        array_index = len(patch_embeddings)

        patch_embeddings.append(patches)

        metadata_records.append({
            'array_index': array_index,
            'filename': row['filename'],
            'label': row['ground_truth'],
            'site': row['site'],
            'filepath': str(filepath)
        })

    except Exception as error:
        print(
            f"ERROR: {row['filename']} | {error}"
        )

        error_records.append({
            'filename': row['filename'],
            'site': row['site'],
            'error': str(error)
        })

    if (row_index + 1) % 50 == 0:
        print(
            f"Processed "
            f"{row_index + 1}/{len(test_site_df)}"
        )

### 9.1 Assemble and inspect the test-site outputs

Stack individual image embeddings into a single three-dimensional array and create matching metadata and error tables.


In [ ]:
# Combine per-image arrays and aligned metadata
patch_embeddings = np.stack(
    patch_embeddings,
    axis=0
)

test_metadata_df = pd.DataFrame(
    metadata_records
)

test_errors_df = pd.DataFrame(
    error_records
)

print(
    f"Patch array shape: "
    f"{patch_embeddings.shape}"
)

print(
    f"Metadata rows: "
    f"{len(test_metadata_df)}"
)

print(
    f"Errors: "
    f"{len(test_errors_df)}"
)

### 9.2 Save the test-site outputs

Persist the patch array and its aligned metadata to Google Drive.


In [ ]:
# Cell 13: Save test-site outputs
test_patch_path = (
    Path(PATCH_SAVE_FOLDER)
    / f'dinov3_patches_{test_site}.npy'
)

test_metadata_path = (
    Path(PATCH_SAVE_FOLDER)
    / f'meta_{test_site}.csv'
)

test_error_path = (
    Path(PATCH_SAVE_FOLDER)
    / f'errors_{test_site}.csv'
)

In [ ]:
np.save(
    test_patch_path,
    patch_embeddings
)

test_metadata_df.to_csv(
    test_metadata_path,
    index=False
)

if len(test_errors_df) > 0:
    test_errors_df.to_csv(
        test_error_path,
        index=False
    )

print(f"Saved patches:\n{test_patch_path}")
print(f"Saved metadata:\n{test_metadata_path}")

### 9.3 Reload and verify saved data

Reload the persisted files using memory mapping to confirm that the saved shape, dtype and metadata alignment are intact.


In [ ]:
#Cell 14: Reload and verify test site
loaded_patches = np.load(
    test_patch_path,
    mmap_mode='r'
)

loaded_metadata = pd.read_csv(
    test_metadata_path
)

print(f"Loaded patch shape: {loaded_patches.shape}")
print(f"Loaded patch type: {loaded_patches.dtype}")
print(f"Metadata rows: {len(loaded_metadata)}")

In [ ]:
loaded_patches = np.load(
    test_patch_path,
    mmap_mode='r'
)

loaded_metadata = pd.read_csv(
    test_metadata_path
)

print(f"Loaded patch shape: {loaded_patches.shape}")
print(f"Loaded patch type: {loaded_patches.dtype}")
print(f"Metadata rows: {len(loaded_metadata)}")

### 9.4 Inspect the spatial patch grid

Reshape one image from `(196, 1024)` into its 14 × 14 spatial grid. Each grid position contains a 1,024-dimensional DINOv3 feature vector.


In [ ]:
#Cell 15: Inspect one patch grid
image_index = 0

one_image_patches = loaded_patches[
    image_index
]

patch_grid = one_image_patches.reshape(
    14,
    14,
    1024
)

print(
    f"One image patch matrix: "
    f"{one_image_patches.shape}"
)

print(
    f"Spatial patch grid: "
    f"{patch_grid.shape}"
)

print(
    f"Image represented by: "
    f"{patch_grid.shape[0]} rows × "
    f"{patch_grid.shape[1]} columns"
)

## 10. Extract patch embeddings for all sites

Process each monitoring site independently. Existing outputs are skipped so that an interrupted run can be resumed without recomputing completed sites.


In [ ]:
#Cell 16: Extract every site
sites = sampled_df['site'].unique()

print(f"Sites to process: {len(sites)}")

In [ ]:
for site_number, site in enumerate(sites, start=1):

    patch_save_path = (
        Path(PATCH_SAVE_FOLDER)
        / f'dinov3_patches_{site}.npy'
    )

    metadata_save_path = (
        Path(PATCH_SAVE_FOLDER)
        / f'meta_{site}.csv'
    )

    error_save_path = (
        Path(PATCH_SAVE_FOLDER)
        / f'errors_{site}.csv'
    )

    if (
        patch_save_path.exists()
        and metadata_save_path.exists()
    ):
        print(
            f"\nSKIP {site_number}/{len(sites)}: "
            f"{site}"
        )
        continue

    site_df = (
        sampled_df[
            sampled_df['site'] == site
        ]
        .reset_index(drop=True)
    )

    patch_embeddings = []
    metadata_records = []
    error_records = []

    print(
        f"\nPROCESSING "
        f"{site_number}/{len(sites)}: "
        f"{site}"
    )

    print(f"Images: {len(site_df)}")

    for row_index, row in site_df.iterrows():

        filepath = (
            Path(DATASET_FOLDER)
            / row['site']
            / row['ground_truth']
            / row['filename']
        )

        try:
            patches = extract_patch_embeddings(
                filepath,
                row['site']
            )

            array_index = len(
                patch_embeddings
            )

            patch_embeddings.append(
                patches
            )

            metadata_records.append({
                'array_index': array_index,
                'filename': row['filename'],
                'label': row['ground_truth'],
                'site': row['site'],
                'filepath': str(filepath)
            })

        except Exception as error:
            print(
                f"ERROR: "
                f"{row['filename']} | "
                f"{error}"
            )

            error_records.append({
                'filename': row['filename'],
                'site': row['site'],
                'error': str(error)
            })

        if (row_index + 1) % 50 == 0:
            print(
                f"Processed "
                f"{row_index + 1}/"
                f"{len(site_df)}"
            )

    if len(patch_embeddings) == 0:
        print(
            f"No embeddings extracted "
            f"for {site}"
        )
        continue

    patch_array = np.stack(
        patch_embeddings,
        axis=0
    )

    metadata_df = pd.DataFrame(
        metadata_records
    )

    errors_df = pd.DataFrame(
        error_records
    )

    np.save(
        patch_save_path,
        patch_array
    )

    metadata_df.to_csv(
        metadata_save_path,
        index=False
    )

    if len(errors_df) > 0:
        errors_df.to_csv(
            error_save_path,
            index=False
        )

    print(
        f"Saved patch shape: "
        f"{patch_array.shape}"
    )

    print(
        f"Saved metadata rows: "
        f"{len(metadata_df)}"
    )

    print(
        f"Errors: "
        f"{len(errors_df)}"
    )

    # Release site-level arrays before processing the next site.
    del patch_embeddings, patch_array, metadata_df, errors_df
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()gs
    del patch_array
    del metadata_df

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nPatch extraction complete")

## 11. Validate all saved site outputs

Check that every saved array has the expected `(n_images, 196, 1024)` structure and that its image count matches the corresponding metadata file.


In [ ]:
#Cell 17: Validate every site
validation_records = []

for site in sites:

    patch_path = (
        Path(PATCH_SAVE_FOLDER)
        / f'dinov3_patches_{site}.npy'
    )

    metadata_path = (
        Path(PATCH_SAVE_FOLDER)
        / f'meta_{site}.csv'
    )

    if not patch_path.exists():
        validation_records.append({
            'site': site,
            'status': 'missing patch file',
            'images': None,
            'patches': None,
            'dimensions': None,
            'dtype': None
        })

        continue

    if not metadata_path.exists():
        validation_records.append({
            'site': site,
            'status': 'missing metadata',
            'images': None,
            'patches': None,
            'dimensions': None,
            'dtype': None
        })

        continue

    patches = np.load(
        patch_path,
        mmap_mode='r'
    )

    metadata = pd.read_csv(
        metadata_path
    )

    valid = (
        patches.ndim == 3
        and patches.shape[0] == len(metadata)
        and patches.shape[1] == 196
        and patches.shape[2] == 1024
    )

    validation_records.append({
        'site': site,
        'status': 'valid' if valid else 'check',
        'images': patches.shape[0],
        'patches': patches.shape[1],
        'dimensions': patches.shape[2],
        'dtype': str(patches.dtype)
    })

In [ ]:
validation_df = pd.DataFrame(
    validation_records
)

display(validation_df)

### 11.1 Validation summary

Report the number of valid sites and the total number of images represented by successfully validated patch arrays.


In [ ]:
valid_sites = validation_df[
    validation_df['status'] == 'valid'
]

print(
    f"Valid sites: "
    f"{len(valid_sites)}/{len(sites)}"
)

print(
    f"Total images with patches: "
    f"{valid_sites['images'].sum()}"
)

## 12. Inspect generated files

List the output directory as a final check that the expected embedding and metadata files were created.


In [ ]:
import os

os.listdir(OUTPUT_DIR)